# Prepare dataset

In [2]:
# imports

import os
import torch
import pandas as pd
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer

D:\univ_subj\M_An_I\Sem 2\BioMedical NLP\GutBrainIE\GBIE\.vgbie\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
articles_path = 'D:\\univ_subj\\M_An_I\\Sem 2\\BioMedical NLP\\GutBrainIE\\GutBrainIE_Full_Collection_2026\\Articles\\csv_format'
annotations_path = 'D:\\univ_subj\\M_An_I\\Sem 2\\BioMedical NLP\\GutBrainIE\\GutBrainIE_Full_Collection_2026\\Annotations'

In [4]:
categories_to_ids = {
    'anatomical location': 0,
    'animal': 1,
    'biomedical technique': 2,
    'bacteria': 3,
    'chemical': 4,
    'dietary supplement': 5,
    'DDF': 6,
    'drug': 7,
    'food': 8,
    'gene': 9,
    'human': 10,
    'microbiome': 11,
    'statistical technique': 12,
    'none': 13
}

ids_to_categories = {v: k for k, v in categories_to_ids.items()}

In [53]:
def get_annotations(annotations_file):
    annotations_df = pd.read_csv(annotations_file, sep='|')
    annotations_df = annotations_df.drop(columns=['annotator'])
    annotations_df['location'] = annotations_df['location'].map(lambda x: 1 if x == 'title' else 0)
    annotations_df['label'] = annotations_df['label'].map(lambda x: categories_to_ids[x])
    print(annotations_df.head())
    return annotations_df

def get_articles(articles_file, tokenizer):
    old_articles_df = pd.read_csv(articles_file, sep='|')
    articles_df = old_articles_df[['pmid']].copy()
    old_articles_df['title'] = old_articles_df['title'].map(lambda x: x + '. ')
    texts = old_articles_df['title'] + '. ' + old_articles_df['abstract']
    articles_df['text'] = texts
    articles_df['added_chars_no'] = old_articles_df['title'].map(lambda x: len(x) + 2)
    texts = texts.map(lambda x: tokenizer(x, truncation=True, padding=True, return_offsets_mapping=True))
    articles_df['input_ids'] = texts.map(lambda x: x['input_ids'])
    articles_df['offset_mapping'] = texts.map(lambda x: x['offset_mapping'])
    print(articles_df.head())
    return articles_df


In [66]:
class GBDataset(Dataset):
    def __init__(self, annotations_df, articles_df):
        self.annotations_df = annotations_df
        self.articles_df = articles_df

        self.dataset = []

        mismatches = 0

        for idx in range(self.articles_df.shape[0]):
            start_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            end_labels = torch.zeros(len(self.articles_df["input_ids"][idx]), dtype=torch.uint8)
            span_idxs = []
            span_labels = []
            pmid = self.articles_df["pmid"][idx]
            for row in self.annotations_df[self.annotations_df['pmid'] == pmid].itertuples(name=None):
                start_token = None
                end_token = None

                ann_start_idx = row[2]
                ann_end_idx = row[3]
                # if the entity is in the abstract, shift the idx with the number of chars
                # added at the start of the string for the title
                if row[4] == 0:
                    ann_start_idx += self.articles_df['added_chars_no'][idx]
                    ann_end_idx += self.articles_df['added_chars_no'][idx]

                for i, (offset_start, offset_end) in enumerate(self.articles_df["offset_mapping"][idx]):
                    # Skip special tokens like [CLS] or [SEP] which have (0,0) offsets
                    if offset_start == 0 and offset_end == 0:
                        continue

                    # # Find the first token that starts at or after the char start
                    # if start_token is None and offset_start >= ann_start_idx:
                    #     start_token = i
                    #
                    # # Find the last token that ends at or before the char end
                    # if start_token is not None and offset_end <= ann_end_idx:
                    #     end_token = i
                    #
                    # if offset_end > ann_start_idx and offset_start < ann_end_idx:
                    #     if start_token is None:
                    #         start_token = i
                    #     end_token = i

                    # 1. Capture the start: First token that contains char_start
                    if start_token is None and offset_start <= ann_start_idx < offset_end:
                        start_token = i

                    # 2. Capture the end: Token that contains the last character
                    if offset_start <= ann_end_idx < offset_end:
                        end_token = i


                if start_token is not None and end_token is not None:
                    start_labels[start_token] = 1
                    end_labels[end_token] = 1
                    span_idxs.append([start_token, end_token])
                    span_labels.append(row[6])

                    # 1. Get the original text for comparison
                    original_entity = self.articles_df['text'][idx][ann_start_idx:ann_end_idx + 1]

                    # 2. Get the tokens from the input_ids
                    # input_ids is the tensor/list from the tokenizer encoding
                    token_ids = self.articles_df['input_ids'][idx][start_token : end_token + 1]
                    decoded_entity = tokenizer.decode(token_ids).strip()


                    # 3. Print and compare
                    if original_entity.lower() != decoded_entity.lower():
                        print(f"Mismatch found!")
                        print(f"Original: '{original_entity}'")
                        print(f"Decoded:  '{decoded_entity}'")
                        print(f"Indices:  {start_token}:{end_token}")
                        mismatches += 1

            self.dataset.append((pmid, self.annotations_df[self.annotations_df['pmid'] == pmid]['location'], self.articles_df.iloc[idx]['input_ids'], start_labels, end_labels, span_idxs, span_labels))
        print('mismatches:', mismatches)

    def __len__(self):
        return self.articles_df.shape[0]

    def __getitem__(self, idx):
        return self.dataset[idx]


In [11]:
tokenizer = AutoTokenizer.from_pretrained("thomas-sounack/BioClinical-ModernBERT-base")

In [68]:
bronze_annotations = get_annotations(os.path.join(annotations_path, 'Train/bronze_quality/csv_format/train_bronze_entities.csv'))
bronze_articles = get_articles(os.path.join(articles_path, 'articles_train_bronze.csv'), tokenizer)
bronze_dataset = GBDataset(bronze_annotations, bronze_articles)
torch.save(bronze_dataset, os.path.join('./datasets', 'bronze_dataset.pt'))

       pmid  start_idx  end_idx  location                          text_span  \
0  35017199          0       13         1                     Gut microbiota   
1  35017199         24       54         1    Alzheimer's disease pathologies   
2  35017199         60       78         1                cognitive disorders   
3  35017199         84      116         1  PUFA-associated neuroinflammation   
4  35017199         41       53         0                      gut dysbiosis   

   label  
0     11  
1      6  
2      6  
3      6  
4      6  
       pmid                                               text  \
0  35017199  Gut microbiota regulate Alzheimer's disease pa...   
1  37934614  Gut microbiome-targeted therapies for Alzheime...   
2  31753762  The gut microbiome in neurological disorders.....   
3  28372330  The Gut Microbiota and Alzheimer's Disease.. ....   
4  31118068  Antibiotics, gut microbiota, and Alzheimer's d...   

   added_chars_no                                       

In [69]:
silver_annotations = get_annotations(os.path.join(annotations_path, 'Train/silver_quality/csv_format/train_silver_entities.csv'))
silver_articles = get_articles(os.path.join(articles_path, 'articles_train_silver.csv'), tokenizer)
silver_dataset = GBDataset(silver_annotations, silver_articles)
torch.save(silver_dataset, os.path.join('./datasets', 'silver_dataset.pt'))

       pmid  start_idx  end_idx  location  \
0  27176462          0       13         1   
1  27176462         18       51         1   
2  27176462         19       53         0   
3  27176462         77       96         0   
4  27176462        150      189         0   

                                  text_span  label  
0                            Gut microbiota     11  
1        early pediatric multiple sclerosis      6  
2       gut microbial community composition     11  
3                      neurological disease      6  
4  early onset pediatric multiple sclerosis      6  
       pmid                                               text  \
0  27176462  Gut microbiota in early pediatric multiple scl...   
1  33796919  Bifidobacterium Lactis Probio-M8 regulates gut...   
2  39478626  Characterization of gut microbiota dynamics in...   
3  29065369  The fecal microbiome of ALS patients.. . Amyot...   
4  39224586  Relationship between gut microbiota and multip...   

   added_chars

In [70]:
silver2025_annotations = get_annotations(os.path.join(annotations_path, 'Train/silver_quality/csv_format/train_silver_2025_entities.csv'))
silver2025_articles = get_articles(os.path.join(articles_path, 'articles_train_silver_2025.csv'), tokenizer)
silver2025_dataset = GBDataset(silver2025_annotations, silver2025_articles)
torch.save(silver2025_dataset, os.path.join('./datasets', 'silver2025_dataset.pt'))

       pmid  start_idx  end_idx  location              text_span  label
0  36479494          0       18         1    Dietary supplements      5
1  36479494         23       43         1  neurological diseases      6
2  36479494         49       59         1            brain aging      6
3  36479494          2       13         0           healthy diet      5
4  36479494         81       92         0           brain health      6
       pmid                                               text  \
0  36479494  Dietary supplements in neurological diseases a...   
1  38891970  Human Gut Microbiota for Diagnosis and Treatme...   
2  33381005  Fisetin Regulates Gut Microbiota and Exerts Ne...   
3  38173790  Multidirectional associations between the gut ...   
4  35263739  Is There Any Link between Cognitive Impairment...   

   added_chars_no                                          input_ids  \
0              65  [50281, 37, 2880, 552, 26434, 275, 20967, 6578...   
1              67  [50281, 

In [67]:
gold_annotations = get_annotations(os.path.join(annotations_path, 'Train/gold_quality/csv_format/train_gold_entities.csv'))
gold_articles = get_articles(os.path.join(articles_path, 'articles_train_gold.csv'), tokenizer)
gold_dataset = GBDataset(gold_annotations, gold_articles)
torch.save(gold_dataset, os.path.join('./datasets', 'gold_dataset.pt'))

       pmid  start_idx  end_idx  location                      text_span  \
0  38860943          0        9         1                     Probiotics   
1  38860943         15       35         1          microbial metabolites   
2  38860943        144      162         1            TDP43 mutation mice   
3  38860943          0       28         0  Amyotrophic lateral sclerosis   
4  38860943         31       33         0                            ALS   

   label  
0      5  
1      4  
2      1  
3      6  
4      6  
       pmid                                               text  \
0  38860943  Probiotics and microbial metabolites maintain ...   
1  37074710  Interplay of Metabolome and Gut Microbiome in ...   
2  36093178  A comparison of the composition and functions ...   
3  35328673  The Inflammatory Conspiracy in Multiple Sclero...   
4  40258729  Characterization of the gut microbiome in Alzh...   

   added_chars_no                                          input_ids  \
0       

In [71]:
dev_annotations = get_annotations(os.path.join(annotations_path, 'Dev/csv_format/dev_entities.csv'))
dev_articles = get_articles(os.path.join(articles_path, 'articles_dev.csv'), tokenizer)
dev_dataset = GBDataset(dev_annotations, dev_articles)
torch.save(dev_dataset, os.path.join('./datasets', 'dev_dataset.pt'))

       pmid  start_idx  end_idx  location  \
0  35766370         26       49         1   
1  35766370         55       84         1   
2  35766370         89      124         1   
3  35766370         15       28         0   
4  35766370         98      113         0   

                              text_span  label  
0              Gut Microbiome Dysbiosis      6  
1        Intestinal Barrier Dysfunction      6  
2  Prodromal Alzheimer Disease Patients     10  
3                        gut microbiota     11  
4                      elderly patients     10  
       pmid                                               text  \
0  35766370  Orthopedic Surgery Causes Gut Microbiome Dysbi...   
1  34912029  Cadmium exposure modulates the gut-liver axis ...   
2  40791596  Exploring the mycobiota in multiple sclerosis:...   
3  40376196  Potential benefits of kefir and its compounds ...   
4  36731694  In-depth investigation of the mechanisms of Sc...   

   added_chars_no                     

In [ ]:
# def refine_offsets(predicted_start_char, predicted_end_char, full_text):
#     entity_text = full_text[predicted_start_char:predicted_end_char]
#
#     # Remove leading/trailing whitespace and punctuation
#     # This ensures "spp.," becomes "spp."
#     cleaned_text = entity_text.strip(" ,.;:-()[]")
#
#     # Recalculate offsets based on the cleaned string
#     new_start = full_text.find(cleaned_text, predicted_start_char)
#     new_end = new_start + len(cleaned_text)
#
#     return new_start, new_end